# QStorePrice AI — Complete End-to-End Kaggle Pipeline

**Project**: Fine-tuned Gemma 4 model that writes *Operating Briefs* to manage perishable goods in quick-commerce stores, cutting food waste in Indian dark stores. Unified metric: **WRR (Weekly Waste Recovery Rate)**.

**Hackathon**: Submission for *The Gemma 4 Good Hackathon* — Impact Track: **Global Resilience** · Special Tech Track: **Unsloth**.

**Pipeline**: GitHub clone → Install → Env verify → SFT → GRPO rollouts → DPO → Eval → Backend server → Push to HF Hub

**Hardware target**: Kaggle T4 (16 GB VRAM) or P100 (16 GB VRAM)

**HF credits**: Used for (1) downloading gated Gemma 4 weights (set HF_TOKEN with the Gemma EULA accepted), (2) HF Inference API during GRPO rollouts (optional, faster) and (3) pushing the trained model to HF Hub.

---
### ⚙️ Configuration — Change these before running
| Variable | Default | Notes |
|---|---|---|
| `HF_TOKEN` | required | Your HF token (Gemma 4 weights are gated; accept terms first) |
| `MODEL_ID` | `google/gemma-4-e4b-it` | Use `google/gemma-4-26b-it` for better quality (slower, A100) |
| `USE_HF_INFERENCE_API` | `False` | `True` = use HF credits for GRPO inference |
| `GRPO_EPISODES` | `5` | Increase for better training (slow) |
| `HF_REPO_ID` | set below | Where to push your trained model |

In [ ]:
# ============================================================
# CELL 1 — USER CONFIGURATION
# All numeric knobs default to AUTO. The notebook detects:
#   - available GPU VRAM (T4 vs A100 vs CPU)
#   - dataset size (90 vs 270 vs 450 examples)
# and picks epochs / batch / sequence length accordingly.
# Override any value below to pin it manually.
# ============================================================

import os

# --- Hugging Face ---
# Gemma 4 weights are gated. Accept the EULA at
# https://huggingface.co/google/gemma-4-e4b-it then drop your token here
# (or in a Kaggle Secret labelled HF_TOKEN).
HF_TOKEN      = "hf_REPLACE_WITH_YOUR_TOKEN"
HF_REPO_ID    = "your-hf-username/qstoreprice-sft"
HF_TOKEN_SET  = bool(HF_TOKEN) and not HF_TOKEN.startswith("hf_REPLACE")

# --- Model ---
MODEL_ID = "google/gemma-4-e4b-it"   # E4B fits T4; switch to "google/gemma-4-26b-it" on A100

# --- Auto-tuning master switch ---
# True  = pick all training hyperparameters from VRAM + dataset size detected at runtime.
# False = use the manual *_MANUAL values below.
AUTO_TUNE = True

# --- SFT (manual overrides; used only when AUTO_TUNE=False) ---
SFT_EPOCHS_MANUAL              = 5
SFT_LR_MANUAL                  = 1e-4
SFT_GRAD_ACCUM_MANUAL          = 4
SFT_MAX_SEQ_LEN_MANUAL         = 2048
SFT_PACKING_MANUAL             = False
SFT_N_PER_DIFFICULTY_MANUAL    = 30   # 30 * 9 = 270 examples
SFT_WARMUP_RATIO_MANUAL        = 0.10

# --- SFT self-recovery ---
# If after training the sanity-check brief is missing required sections,
# the notebook will retrain once with 2x epochs. Set to 0 to disable.
SFT_AUTO_RETRIES = 1

# --- GRPO rollouts ---
# Default 3 episodes is enough to seed DPO buffer. Bump to 20+ for real curves.
GRPO_EPISODES_MANUAL = 3

# --- DPO ---
# Enabled by default. Skipped automatically if VRAM is too small to safely
# reload the model after GRPO, or if the trajectory buffer is empty.
DPO_ENABLED          = True
DPO_MIN_PAIRS        = 4
DPO_MIN_VRAM_GB      = 12   # T4 has 14.5 GB so DPO runs by default
DPO_LR_MANUAL        = 5e-7

# --- Misc ---
SEED = 42
USE_HF_INFERENCE_API = False

# --- Paths (Kaggle standard) ---
WORK_DIR         = "/kaggle/working"
REPO_DIR         = f"{WORK_DIR}/QStorePrice"
CHECKPOINTS_DIR  = f"{WORK_DIR}/checkpoints"
SFT_DIR          = f"{CHECKPOINTS_DIR}/sft_v1"
GRPO_DIR         = f"{CHECKPOINTS_DIR}/grpo_level0"
DPO_DIR          = f"{CHECKPOINTS_DIR}/dpo_round1"
FINAL_DIR        = f"{CHECKPOINTS_DIR}/final"
PLOTS_DIR        = f"{WORK_DIR}/plots"

os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# These will be filled in by the auto-tune cell below or by manual overrides.
SFT_EPOCHS = SFT_LR = SFT_GRAD_ACCUM = SFT_MAX_SEQ_LEN = None
SFT_PACKING = SFT_N_PER_DIFFICULTY = SFT_WARMUP_RATIO = None
GRPO_EPISODES = None
VRAM_GB = 0.0

# --- Forward-declared state used by later cells ---
eval_results       = {}
episode_results    = []
trajectory_buffer  = None
CURRENT_CHECKPOINT = SFT_DIR
HF_AUTHED          = False
SERVER_PORT        = 8000
SERVER_URL         = f"http://localhost:{SERVER_PORT}"
server_proc        = None
server_up          = False

print("Configuration loaded.")
print(f"  MODEL_ID            : {MODEL_ID}")
print(f"  AUTO_TUNE           : {AUTO_TUNE}")
print(f"  HF_TOKEN_SET        : {HF_TOKEN_SET}")
print(f"  DPO_ENABLED         : {DPO_ENABLED} (skipped automatically if VRAM < {DPO_MIN_VRAM_GB} GB)")
print(f"  SFT_AUTO_RETRIES    : {SFT_AUTO_RETRIES}")
print()
print("Final hyperparameters are picked in the SFT cell after VRAM + dataset are known.")

## Section 1 — Hardware & Environment Check

In [ ]:
# ============================================================
# CELL 2 — GPU / HARDWARE CHECK
# ============================================================

import subprocess, sys, platform

print("=" * 55)
print(" QStorePrice AI — Kaggle Hardware Check")
print("=" * 55)

# Python version
print(f"Python : {sys.version.split()[0]}")
print(f"OS     : {platform.system()} {platform.release()}")

# GPU check
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"PyTorch: {torch.__version__}")
    if cuda_ok:
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU    : {gpu_name}")
        print(f"VRAM   : {vram_gb:.1f} GB")
        if vram_gb < 14:
            print("WARNING: < 14 GB VRAM — training may OOM. Try 1.5B model.")
    else:
        print("GPU    : NOT AVAILABLE — enable GPU accelerator in Kaggle settings")
except ImportError:
    print("PyTorch not yet installed (will be installed below).")

# Disk space
result = subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True)
print("\nDisk usage (working dir):")
print(result.stdout.strip())

# nvidia-smi summary
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                         "--format=csv,noheader"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"\nnvidia-smi: {result.stdout.strip()}")

## Section 2 — Install Dependencies

In [ ]:
# ============================================================
# CELL 3 — INSTALL UNSLOTH (must be first, Kaggle-specific)
# ============================================================

import subprocess, sys

def run(cmd, desc=""):
    print(f"\n>>> {desc or cmd[:80]}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print("\n".join(result.stdout.strip().split("\n")[-5:]))
    if result.returncode != 0:
        print(f"STDERR: {result.stderr.strip()[-400:]}")
        raise RuntimeError(f"Command failed: {cmd}")
    return result

# Step 1: Fix huggingface_hub FIRST.
# The Kaggle base image ships internally inconsistent huggingface_hub files:
# _snapshot_download.py imports KernelInfo, but hf_api.py doesn't export it.
# Force-reinstalling all files from the same release makes the package consistent.
# --no-deps leaves torch/transformers untouched.
run(
    'pip install -q --upgrade --force-reinstall --no-deps "huggingface_hub>=0.26.0"',
    "Reinstalling huggingface_hub to a self-consistent version"
)

# Evict any cached huggingface_hub sub-modules that were loaded before this fix.
# This matters when Cell 3 is re-run mid-session after other cells already imported it.
for _mod in list(sys.modules):
    if _mod.startswith("huggingface_hub"):
        del sys.modules[_mod]

# Step 2: Install Unsloth with Kaggle-optimized build.
# This runs after huggingface_hub is consistent, so unsloth's own HF Hub
# calls will work without hitting the KernelInfo ImportError.
run(
    'pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"',
    "Installing Unsloth (Kaggle build)"
)

# Step 3: Upgrade unsloth_zoo to match the installed huggingface_hub.
# --no-deps avoids accidentally downgrading torch or transformers.
run(
    "pip install -q --upgrade --no-cache-dir --no-deps unsloth_zoo",
    "Upgrading unsloth_zoo"
)

# Verify the fix took effect before any downstream cell imports unsloth.
try:
    from huggingface_hub import hf_api as _hf_api
    assert hasattr(_hf_api, "KernelInfo"), "KernelInfo still missing from hf_api"
    print("\nhuggingface_hub: KernelInfo present — fix verified.")
    del _hf_api
except Exception as _e:
    print(f"\nWARNING: huggingface_hub fix may not have taken effect: {_e}")
    print("Restart the kernel (Runtime → Restart) and re-run all cells from the top.")

print("\nUnsloth installed successfully.")


In [ ]:
# ============================================================
# CELL 4 — INSTALL REMAINING DEPENDENCIES
# ============================================================

deps = [
    # Core RL + training stack
    "transformers>=4.40.0",
    "trl>=0.8.6",
    "peft>=0.10.0",
    "accelerate>=0.29.3",
    "bitsandbytes>=0.43.1",
    "datasets>=2.19.0",
    # Environment
    "gymnasium>=0.29.1,<1.0",
    "pydantic>=2.0.0",
    # Experiment tracking
    "wandb>=0.17.0",
    # HF Hub utilities
    "huggingface_hub>=0.22.0",
    # Utilities
    "numpy>=1.26.0",
    "pandas>=2.2.1",
    "scipy>=1.13.0",
    "tqdm>=4.66.2",
    "python-dotenv>=1.0.1",
    "matplotlib>=3.7.0",
    # Backend server
    "fastapi>=0.110.0",
    "uvicorn>=0.27.0",
    "httpx>=0.27.0",  # async HTTP client for server testing
]

import subprocess

pkg_str = " ".join(f'"{d}"' for d in deps)
result = subprocess.run(
    f"pip install -q {pkg_str}",
    shell=True, capture_output=True, text=True
)
last = result.stdout.strip().split("\n")[-3:]
print("\n".join(last))
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])
    raise RuntimeError("Dependency installation failed")

# OpenEnv core — used as the deployment / contract layer for the live demo Space.
oe = subprocess.run("pip install -q openenv-core>=0.2.0", shell=True,
                    capture_output=True, text=True)
if oe.returncode == 0:
    print("openenv-core installed.")
else:
    print("openenv-core not available on PyPI — server section will use fallback mode.")

print("\nAll dependencies installed.")

## Section 3 — Clone Repository & Setup

In [ ]:
# ============================================================
# CELL 5 — CLONE REPO AND SET UP PYTHON PATH
# ============================================================

import os, sys, subprocess

WORK_DIR  = "/kaggle/working"
REPO_DIR  = f"{WORK_DIR}/QStorePrice"

# Clone (or pull if already present — handles re-runs)
if os.path.exists(REPO_DIR):
    print(f"Repo already cloned at {REPO_DIR}. Pulling latest...")
    r = subprocess.run("git pull", cwd=REPO_DIR, capture_output=True, text=True, shell=True)
    print(r.stdout.strip() or "Already up to date.")
else:
    print("Cloning QStorePrice repository...")
    r = subprocess.run(
        "git clone https://github.com/nandeshkanagaraju/QStorePrice.git",
        cwd=WORK_DIR, capture_output=True, text=True, shell=True
    )
    if r.returncode != 0:
        raise RuntimeError(f"Clone failed: {r.stderr}")
    print("Clone complete.")

# Add repo root to Python path so all internal imports resolve
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Verify key directories exist
for subdir in ["freshprice_env", "training", "eval", "server", "training/sft_data"]:
    path = os.path.join(REPO_DIR, subdir)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  {subdir:30s} [{status}]")

# Print commit hash for reproducibility
r = subprocess.run("git log --oneline -3", cwd=REPO_DIR,
                   capture_output=True, text=True, shell=True)
print(f"\nRecent commits:\n{r.stdout.strip()}")
print(f"\nRepo root in sys.path: {REPO_DIR}")

## Section 4 — Hugging Face Authentication

In [ ]:
# ============================================================
# CELL 6 — HF AUTHENTICATION (graceful)
# Gemma 4 weights are gated on the Hub, so a valid HF_TOKEN is REQUIRED.
# If your token is missing, the SFT cell will fail to download the base
# model — accept the Gemma EULA at
# https://huggingface.co/google/gemma-4-e4b-it then come back here.
# ============================================================
# Option A: Token is set manually above in CELL 1.
# Option B: Use Kaggle Secrets (recommended).
#   1. Kaggle sidebar -> Add-ons -> Secrets -> Add "HF_TOKEN"
#   2. Uncomment the two lines below.

# from kaggle_secrets import UserSecretsClient
# HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
# HF_TOKEN_SET = True

import os

if not HF_TOKEN_SET:
    print("HF_TOKEN is the placeholder — Gemma 4 weights cannot be downloaded.")
    print("Set HF_TOKEN in Cell 1 (or via Kaggle Secrets) and re-run.")
    HF_AUTHED = False
else:
    try:
        from huggingface_hub import login, whoami
        login(token=HF_TOKEN, add_to_git_credential=False)
        user_info = whoami(token=HF_TOKEN)
        print(f"Logged in as: {user_info['name']}")
        os.environ["HF_TOKEN"] = HF_TOKEN
        os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
        HF_AUTHED = True
    except Exception as e:
        print(f"HF login failed: {e}. Continuing without auth.")
        HF_AUTHED = False

# Disable WandB telemetry
os.environ["WANDB_DISABLED"] = "true"
print(f"HF_AUTHED = {HF_AUTHED}")

## Section 5 — Environment Smoke Test

In [ ]:
# ============================================================
# CELL 7 — FRESHPRICE ENVIRONMENT SMOKE TEST
# Verifies all modules import and the Gym env boots correctly.
# Expected: reset() returns a 1000-5000 char observation string.
# ============================================================

import sys, os
sys.path.insert(0, REPO_DIR)  # safety: ensure path is set

print("Importing FreshPriceEnv...")
from freshprice_env.freshprice_env import FreshPriceEnv
from freshprice_env.enums import CurriculumScenario
from freshprice_env.constants import TOTAL_TICKS, BRIEFS_PER_EPISODE, TICKS_PER_BRIEF

print("\n--- Episode Constants ---")
print(f"  Total ticks/episode : {TOTAL_TICKS}")
print(f"  Briefs/episode      : {BRIEFS_PER_EPISODE}")
print(f"  Ticks per brief     : {TICKS_PER_BRIEF}")
print(f"  Curriculum scenarios: {[s.name for s in CurriculumScenario]}")

# Test each scenario initialises
print("\n--- Scenario Reset Tests ---")
for scenario in CurriculumScenario:
    env = FreshPriceEnv(scenario=scenario, seed=42)
    obs, info = env.reset()
    state = env.state()
    print(
        f"  {scenario.name:20s} | obs_len={len(obs):5d} | "
        f"batches={state['active_batches']} | "
        f"engine={info['engine_type']}"
    )
    assert len(obs) > 100, f"Observation too short for {scenario.name}"

# Run a few manual steps on STABLE_WEEK with a hardcoded brief
print("\n--- Manual 3-Step Walkthrough (STABLE_WEEK) ---")
env = FreshPriceEnv(scenario=CurriculumScenario.STABLE_WEEK, seed=42)
obs, info = env.reset()
print(f"Initial engine: {info['engine_type']}")
print(f"Observation excerpt (first 400 chars):\n{obs[:400]}")

# A minimal but valid Operating Brief
FALLBACK_BRIEF = """\
## SITUATION SUMMARY
Reviewing current inventory for pricing optimization.

## SIGNAL ANALYSIS
Some batches approaching expiry. Moderate demand velocity.

## VIABILITY CHECK
Small discounts on near-expiry stock should recover cost.

## RECOMMENDATION
Apply conservative discounts to URGENT batches.

## DIRECTIVE
{"engine": "PRICING", "actions": []}

## CONFIDENCE
0.6
"""

total_reward = 0.0
for step in range(3):
    obs, reward, done, truncated, info = env.step(FALLBACK_BRIEF)
    total_reward += reward
    print(
        f"  Step {step+1}: reward={reward:+.4f} | "
        f"parse={'OK' if info['parse_success'] else 'FAIL'} | "
        f"quality={info['quality_score']:.3f} | "
        f"next_engine={info.get('next_engine_type','END')}"
    )

print(f"\n3-step cumulative reward: {total_reward:+.4f}")
print("\nEnvironment smoke test PASSED.")

## Section 5b — Submission Validation
Quick health check: imports, env resets, server admin routes, static files, SFT generator.

In [ ]:
# ============================================================
# CELL 7b — VALIDATE SUBMISSION
# Runs validate_submission.py to confirm everything wired up correctly.
# Failure here means the rest of the notebook will break too.
# ============================================================

import subprocess, sys

result = subprocess.run(
    [sys.executable, "validate_submission.py"],
    cwd=REPO_DIR, capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-800:])
    print("\n[!] Validation reported failures. Inspect output above.")
else:
    print("\nValidation OK.")


## Section 6 — SFT Data Generation & Inspection


In [ ]:
# ============================================================
# CELL 8a — GENERATE SFT TRAINING DATA (auto-sized)
# Picks dataset size based on AUTO_TUNE and the model. Larger models need
# fewer examples to learn the 6-section format; smaller models need more.
# Override SFT_N_PER_DIFFICULTY_MANUAL in Cell 1 to pin the count.
# ============================================================

import sys, os, torch
sys.path.insert(0, REPO_DIR)

# ---- Decide how many examples per (engine, difficulty) ----
if AUTO_TUNE:
    mid = MODEL_ID.lower()
    if "1.5b" in mid or "1b" in mid or "0.5b" in mid:
        SFT_N_PER_DIFFICULTY = 30   # 270 examples — needed for small model to learn format
    elif "7b" in mid:
        SFT_N_PER_DIFFICULTY = 20   # 180 examples — 7B picks up format faster
    elif "14b" in mid or "13b" in mid or "32b" in mid or "70b" in mid:
        SFT_N_PER_DIFFICULTY = 15   # 135 examples — large models barely need any
    else:
        SFT_N_PER_DIFFICULTY = 25   # safe default
    print(f"AUTO_TUNE: SFT_N_PER_DIFFICULTY = {SFT_N_PER_DIFFICULTY} (model={MODEL_ID})")
else:
    SFT_N_PER_DIFFICULTY = SFT_N_PER_DIFFICULTY_MANUAL
    print(f"MANUAL: SFT_N_PER_DIFFICULTY = {SFT_N_PER_DIFFICULTY}")

total_examples = SFT_N_PER_DIFFICULTY * 9  # 3 engines x 3 difficulties
print(f"Total SFT examples to generate: {total_examples}")

# ---- Generate ----
from training.generate_sft_data import generate_all
sft_data_dir = os.path.join(REPO_DIR, "training", "sft_data")
generate_all(output_dir=sft_data_dir, n_per_difficulty=SFT_N_PER_DIFFICULTY)

# ---- Detect VRAM here so the SFT cell can pick batch size / seq len ----
if torch.cuda.is_available():
    VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    print(f"\nDetected VRAM: {VRAM_GB} GB on {torch.cuda.get_device_name(0)}")
else:
    VRAM_GB = 0.0
    print("\nNo CUDA GPU detected — training will be very slow on CPU.")


In [ ]:
# ============================================================
# CELL 8b — INSPECT SFT TRAINING DATA
# Verifies all 90 examples were written and have the 6 required
# section headers. Runs after CELL 8a.
# ============================================================

import json
from pathlib import Path

sft_data_dir = Path(REPO_DIR) / "training" / "sft_data"
print("SFT data files:")

all_examples = []
if not sft_data_dir.exists():
    print(f"  [!] Directory not found: {sft_data_dir}")
    print("  Run CELL 8a first to generate the data.")
else:
    for json_file in sorted(sft_data_dir.glob("*.json")):
        with open(json_file) as f:
            examples = json.load(f)
        all_examples.extend(examples)
        engine_counts = {}
        for ex in examples:
            et = ex.get("engine_type", "UNKNOWN")
            engine_counts[et] = engine_counts.get(et, 0) + 1
        print(f"  {json_file.name:30s} {len(examples):3d} examples | {engine_counts}")

print(f"\nTotal SFT examples: {len(all_examples)}")

difficulty = {}
for ex in all_examples:
    d = ex.get("difficulty", "unknown")
    difficulty[d] = difficulty.get(d, 0) + 1
print(f"Difficulty split: {difficulty}")

if not all_examples:
    print("\n[!] No examples loaded — skipping display and validation.")
else:
    print("\n--- Example #1 (truncated) ---")
    ex = all_examples[0]
    print(f"Engine type : {ex.get('engine_type')}")
    print(f"Difficulty  : {ex.get('difficulty')}")
    print(f"Prompt len  : {len(ex.get('prompt', ''))} chars")
    print(f"Completion  : {ex.get('completion', '')[:400]}...")

    REQUIRED = ["SITUATION:", "SIGNAL ANALYSIS:", "VIABILITY CHECK:",
                "RECOMMENDATION:", "DIRECTIVE:", "CONFIDENCE:"]
    bad = []
    for i, ex in enumerate(all_examples):
        comp = ex.get("completion", "")
        missing = [s for s in REQUIRED if s not in comp]
        if missing:
            bad.append((i, missing))

    if bad:
        print(f"\nWARNING: {len(bad)} examples missing required sections:")
        for idx, miss in bad[:5]:
            print(f"  Example {idx}: missing {miss}")
    else:
        print(f"\nAll {len(all_examples)} examples contain all 6 required sections.")


## Section 7 — SFT Warm-Start Training

In [ ]:
# ============================================================
# CELL 9 — RUN SFT WARM-START (adaptive)
# Picks epochs/batch/seq-len from VRAM_GB and dataset size, then trains.
# If the post-train sanity check fails (model didn\'t learn 6-section
# format), automatically retrains once with 2x epochs (controlled by
# SFT_AUTO_RETRIES in Cell 1).
# ============================================================

import os, sys, math, gc, torch
sys.path.insert(0, REPO_DIR)

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

from training.sft_trainer import load_sft_dataset
from freshprice_env.brief_pipeline.prompt_builder import OperatingBriefPromptBuilder

# ---------------------------------------------------------------
# 1. Load dataset (we need its size to scale epochs)
# ---------------------------------------------------------------
dataset = load_sft_dataset(data_dir=os.path.join(REPO_DIR, "training", "sft_data"))
n_examples = len(dataset)

# ---------------------------------------------------------------
# 2. Pick hyperparameters (AUTO_TUNE branch vs manual)
# ---------------------------------------------------------------
if AUTO_TUNE:
    # ---- VRAM tier -> seq len, batch, grad accum ----
    if VRAM_GB == 0:
        # CPU: tiny config, training will be slow regardless
        MAX_SEQ_LEN, GRAD_ACCUM, PER_DEVICE_BS = 1024, 8, 1
        tier = "cpu"
    elif VRAM_GB < 10:
        MAX_SEQ_LEN, GRAD_ACCUM, PER_DEVICE_BS = 1024, 8, 1
        tier = "<10GB"
    elif VRAM_GB < 16:
        # T4 / similar
        MAX_SEQ_LEN, GRAD_ACCUM, PER_DEVICE_BS = 2048, 4, 1
        tier = "T4-class"
    elif VRAM_GB < 24:
        # 3090 / V100 / A10
        MAX_SEQ_LEN, GRAD_ACCUM, PER_DEVICE_BS = 3072, 2, 1
        tier = "3090-class"
    else:
        # A100 / H100
        MAX_SEQ_LEN, GRAD_ACCUM, PER_DEVICE_BS = 4096, 2, 2
        tier = "A100-class"

    # ---- Dataset size -> epoch count ----
    if n_examples < 150:
        SFT_EPOCHS = 8         # tiny dataset needs more passes
    elif n_examples < 350:
        SFT_EPOCHS = 5
    elif n_examples < 600:
        SFT_EPOCHS = 4
    else:
        SFT_EPOCHS = 3

    SFT_LR             = 1e-4
    SFT_PACKING        = False
    SFT_WARMUP_RATIO   = 0.10
    print(f"AUTO_TUNE picked:")
    print(f"  VRAM tier         : {tier} ({VRAM_GB} GB)")
    print(f"  Dataset size      : {n_examples} examples")
    print(f"  MAX_SEQ_LEN       : {MAX_SEQ_LEN}")
    print(f"  GRAD_ACCUM        : {GRAD_ACCUM}")
    print(f"  PER_DEVICE_BS     : {PER_DEVICE_BS}")
    print(f"  SFT_EPOCHS        : {SFT_EPOCHS}")
    print(f"  SFT_LR            : {SFT_LR}")
else:
    SFT_EPOCHS         = SFT_EPOCHS_MANUAL
    SFT_LR             = SFT_LR_MANUAL
    GRAD_ACCUM         = SFT_GRAD_ACCUM_MANUAL
    MAX_SEQ_LEN        = SFT_MAX_SEQ_LEN_MANUAL
    PER_DEVICE_BS      = 1
    SFT_PACKING        = SFT_PACKING_MANUAL
    SFT_WARMUP_RATIO   = SFT_WARMUP_RATIO_MANUAL
    print("Using manual SFT hyperparameters.")

REQUIRED_SECTIONS = ["SITUATION:", "SIGNAL ANALYSIS:", "VIABILITY CHECK:",
                     "RECOMMENDATION:", "DIRECTIVE:", "CONFIDENCE:"]

def build_model_and_train(epochs_to_use):
    """Load fresh model, attach LoRA, and train. Returns (model, tokenizer, training_loss)."""
    print(f"\nLoading {MODEL_ID} in 4-bit with Unsloth...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=True,
        token=HF_TOKEN if HF_TOKEN_SET else None,
    )
    total = sum(p.numel() for p in model.parameters())
    print(f"Model loaded. Total params: {total/1e6:.1f}M")

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16, lora_dropout=0, bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  LoRA trainable: {trainable/1e6:.2f}M ({trainable/total*100:.1f}% of total)")

    effective_bs = PER_DEVICE_BS * GRAD_ACCUM
    total_steps  = max(1, math.ceil(n_examples / effective_bs) * epochs_to_use)
    warmup_steps = max(1, int(SFT_WARMUP_RATIO * total_steps))
    print(f"  Effective batch  : {effective_bs}")
    print(f"  Estimated steps  : {total_steps} ({epochs_to_use} epochs)")
    print(f"  Warmup steps     : {warmup_steps}")

    os.makedirs(SFT_DIR, exist_ok=True)
    args = SFTConfig(
        output_dir=SFT_DIR,
        num_train_epochs=epochs_to_use,
        per_device_train_batch_size=PER_DEVICE_BS,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=SFT_LR,
        warmup_steps=warmup_steps,
        lr_scheduler_type="cosine",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        save_steps=500,
        seed=SEED,
        max_seq_length=MAX_SEQ_LEN,
        dataset_text_field="text",
        report_to="none",
        packing=SFT_PACKING,
    )
    trainer = SFTTrainer(model=model, tokenizer=tokenizer,
                         train_dataset=dataset, args=args)
    torch.cuda.empty_cache()
    print("\nTraining...")
    stats = trainer.train()
    print(f"  Training loss : {stats.training_loss:.4f}")
    print(f"  Runtime       : {stats.metrics.get('train_runtime', 0):.1f}s")
    return model, tokenizer, stats.training_loss

def sanity_check_brief(model, tokenizer):
    """Generate a brief and return list of missing sections (empty = good)."""
    FastLanguageModel.for_inference(model)
    test_prompt = (
        f"<|im_start|>system\n{OperatingBriefPromptBuilder.SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        "=== CURRENT INVENTORY ===\n"
        "[\U0001f534] dairy \u2014 Batch B001\n"
        "  Stock: 8 units | Expiry: 4.0hrs | Urgency: CRITICAL\n"
        "  Current price: Rs 80 | Original: Rs 80 | Floor: Rs 35\n"
        "=== YOUR TASK ===\nWrite a PRICING Operating Brief.\n"
        "<|im_end|>\n<|im_start|>assistant\n"
    )
    inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True,
                       max_length=MAX_SEQ_LEN).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=500, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                 skip_special_tokens=True)
    missing = [s for s in REQUIRED_SECTIONS if s not in generated]
    return generated, missing

# ---------------------------------------------------------------
# 3. Train + sanity check + auto-retry
# ---------------------------------------------------------------
attempt = 0
epochs_for_attempt = SFT_EPOCHS
model = tokenizer = None
final_loss = None

while attempt <= SFT_AUTO_RETRIES:
    attempt += 1
    label = "INITIAL" if attempt == 1 else f"RETRY #{attempt-1}"
    print(f"\n========== SFT attempt {attempt} ({label}, {epochs_for_attempt} epochs) ==========")
    if model is not None:
        try:
            del model, tokenizer
        except Exception:
            pass
        gc.collect()
        torch.cuda.empty_cache()
    model, tokenizer, final_loss = build_model_and_train(epochs_for_attempt)

    print("\nSaving merged 16-bit checkpoint...")
    model.save_pretrained_merged(SFT_DIR, tokenizer, save_method="merged_16bit")

    print("\nGeneration sanity check...")
    generated, missing = sanity_check_brief(model, tokenizer)
    print("Generated brief excerpt:")
    print(generated[:600])

    if not missing and final_loss < 1.5:
        print(f"\nAll {len(REQUIRED_SECTIONS)} sections present. Loss={final_loss:.3f}. SFT VERIFIED.")
        break

    if attempt > SFT_AUTO_RETRIES:
        print(f"\n[!] After {attempt} attempt(s), missing={missing}, final_loss={final_loss:.3f}")
        print("    Continuing anyway — downstream cells will use whatever the model produces.")
        break

    epochs_for_attempt = max(epochs_for_attempt * 2, 10)
    print(f"\n[!] Sanity check failed: missing={missing}, loss={final_loss:.3f}")
    print(f"    Retrying with {epochs_for_attempt} epochs (SFT_AUTO_RETRIES={SFT_AUTO_RETRIES}).")

CURRENT_CHECKPOINT = SFT_DIR
print(f"\nCurrent checkpoint: {CURRENT_CHECKPOINT}")


## Section 8 — Rollout Collection (historically called "GRPO")

Despite the name, this stage does **not** do gradient updates. It runs the SFT model in the environment to collect trajectories, which seed the DPO preference buffer in the next section. The actual RL gradient step is **DPO** (Section 9).


In [ ]:
# ============================================================
# CELL 10 — DEFINE INFERENCE BACKEND
# Chooses between HF Inference API (uses HF credits) or local GPU.
# Both are wrapped behind a `.generate(prompt: str) -> str` interface so
# FreshPriceEnv can call either transparently.
# ============================================================

import sys
sys.path.insert(0, REPO_DIR)

class LocalInferenceClient:
    """Local GPU inference using the SFT/DPO model already loaded in memory."""
    def __init__(self, model, tokenizer, system_prompt):
        from unsloth import FastLanguageModel
        FastLanguageModel.for_inference(model)
        self._model = model
        self._tok   = tokenizer
        self._sys   = system_prompt

    def generate(self, prompt: str) -> str:
        import torch
        full = (
            f"<|im_start|>system\n{self._sys}<|im_end|>\n"
            f"<|im_start|>user\n{prompt}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        inputs = self._tok(full, return_tensors="pt", truncation=True,
                           max_length=3800).to(self._model.device)
        with torch.no_grad():
            out = self._model.generate(
                **inputs, max_new_tokens=600,
                temperature=0.7, do_sample=True,
                pad_token_id=self._tok.eos_token_id,
            )
        return self._tok.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )


class HFAPIInferenceClient:
    """HF Inference API client — uses your HF credits (~0.05 per episode)."""
    def __init__(self, model_id, token, system_prompt):
        from huggingface_hub import InferenceClient
        self._client = InferenceClient(model=model_id, token=token)
        self._model  = model_id
        self._sys    = system_prompt

    def generate(self, prompt: str) -> str:
        response = self._client.chat_completion(
            messages=[
                {"role": "system", "content": self._sys},
                {"role": "user",   "content": prompt},
            ],
            max_tokens=600,
            temperature=0.7,
        )
        return response.choices[0].message.content


from freshprice_env.brief_pipeline.prompt_builder import OperatingBriefPromptBuilder

if USE_HF_INFERENCE_API:
    print(f"Using HF Inference API: {MODEL_ID}")
    if not HF_TOKEN_SET:
        raise RuntimeError("USE_HF_INFERENCE_API=True requires HF_TOKEN to be set.")
    llm_client = HFAPIInferenceClient(
        model_id=MODEL_ID, token=HF_TOKEN,
        system_prompt=OperatingBriefPromptBuilder.SYSTEM_PROMPT,
    )
else:
    print("Using local GPU inference (free, slower).")
    llm_client = LocalInferenceClient(
        model=model, tokenizer=tokenizer,
        system_prompt=OperatingBriefPromptBuilder.SYSTEM_PROMPT,
    )

test_out = llm_client.generate("=== QUICK TEST ===\nWrite one sentence about pricing perishables.")
print(f"\nTest generation: {test_out[:200]}")
print("\nInference client ready.")


In [ ]:
# ============================================================
# CELL 11 — GRPO EPISODE ROLLOUTS
# Runs GRPO_EPISODES of the LLM acting in FreshPriceEnv. Trajectories feed
# the DPO buffer below. The number of episodes auto-scales with VRAM:
#   < 16 GB (T4):   3 episodes (default)
#   16-24 GB:       6 episodes
#   >= 24 GB:       12 episodes
# Override GRPO_EPISODES_MANUAL in Cell 1 to pin the count.
# ============================================================

import sys, os, json, time, random
sys.path.insert(0, REPO_DIR)

from freshprice_env.freshprice_env import FreshPriceEnv
from freshprice_env.enums import CurriculumScenario
from freshprice_env.monitoring import metrics
from training.curriculum import CurriculumManager, EpisodeResult
from training.trajectory_buffer import Trajectory, TrajectoryBuffer
from training.counterfactual import CounterfactualEngine

# ---- Resolve GRPO_EPISODES ----
if AUTO_TUNE:
    if VRAM_GB < 16:
        GRPO_EPISODES = 3
    elif VRAM_GB < 24:
        GRPO_EPISODES = 6
    else:
        GRPO_EPISODES = 12
    print(f"AUTO_TUNE: GRPO_EPISODES = {GRPO_EPISODES} (VRAM_GB={VRAM_GB})")
else:
    GRPO_EPISODES = GRPO_EPISODES_MANUAL
    print(f"MANUAL: GRPO_EPISODES = {GRPO_EPISODES}")

rng                 = random.Random(SEED)
curriculum          = CurriculumManager()
trajectory_buffer   = TrajectoryBuffer(rng=rng)
counterfactual_eng  = CounterfactualEngine(rng=rng)

episode_results = []
scenario = CurriculumScenario.STABLE_WEEK

print(f"\nStarting GRPO rollouts: {GRPO_EPISODES} episodes on {scenario.name}")
print("-" * 60)
print(f"{'Ep':>4} {'WRR':>6} {'R1-P':>6} {'R2-F':>6} {'R3-T':>6} {'Qual':>6} "
      f"{'Viol':>5} {'Const':>6} {'Time':>6}")
print("-" * 60)

total_start = time.time()

for ep_idx in range(GRPO_EPISODES):
    ep_seed = rng.randint(0, 999999)
    ep_start = time.time()

    try:
        env = FreshPriceEnv(scenario=scenario, seed=ep_seed, llm_client=llm_client)
        obs, info = env.reset(seed=ep_seed)

        ep_briefs   = []
        done        = False
        step_count  = 0

        while not done:
            try:
                raw_brief = llm_client.generate(obs)
            except Exception as gen_err:
                print(f"    [ep {ep_idx+1} step {step_count}] generate failed: {gen_err}")
                raw_brief = (
                    "SITUATION: fallback\n\nSIGNAL ANALYSIS: N/A\n\n"
                    "VIABILITY CHECK: N/A\n\nRECOMMENDATION: hold\n\n"
                    "DIRECTIVE:\n{\"engine\": \"PRICING\", \"actions\": []}\n\n"
                    "CONFIDENCE: LOW\n"
                )
            obs, reward, done, truncated, info = env.step(raw_brief)

            ep_briefs.append({
                "tick":         info.get("tick", step_count * 8),
                "engine_type":  info.get("engine_type", "PRICING"),
                "raw_response": raw_brief,
                "quality_score":info.get("quality_score", 0.0),
                "reward_delta": reward,
                "parse_success":info.get("parse_success", True),
            })

            metrics.record_step(
                scenario=scenario.name,
                tick=info.get("tick", step_count * 8),
                engine_type=info.get("engine_type", "PRICING"),
                reward=float(reward),
                quality_score=float(info.get("quality_score", 0.0)),
                parse_success=bool(info.get("parse_success", True)),
            )
            step_count += 1

        final_reward = info.get("final_reward", {}) or {}
        audit        = info.get("constitutional_audit", {}) or {}

        wrr            = float(final_reward.get("wrr", 0.0))
        r1             = float(final_reward.get("r1_pricing", 0.0))
        r2             = float(final_reward.get("r2_farmer", 0.0))
        r3             = float(final_reward.get("r3_trend", 0.0))
        quality        = float(final_reward.get("brief_quality_score", 0.0))
        violations     = int(final_reward.get("anti_hack_violations", 0))
        valid          = bool(final_reward.get("episode_valid", True))
        const_passed   = bool(audit.get("passed", True))

    except Exception as ep_err:
        print(f"  [!] Episode {ep_idx+1} crashed: {type(ep_err).__name__}: {ep_err}")
        wrr = r1 = r2 = r3 = quality = 0.0
        violations = 0
        valid = const_passed = False
        step_count = 0
        ep_briefs = []
        final_reward = {}

    elapsed = time.time() - ep_start
    ep_result = {
        "episode": ep_idx, "seed": ep_seed, "wrr": wrr,
        "r1": r1, "r2": r2, "r3": r3,
        "quality": quality, "violations": violations,
        "valid": valid, "constitutional_passed": const_passed,
        "steps": step_count, "elapsed_sec": elapsed,
    }
    episode_results.append(ep_result)

    metrics.record_episode(
        scenario=scenario.name, agent_type="llm-grpo",
        wrr=wrr, r1_pricing=r1, r2_farmer=r2, r3_trend=r3,
        brief_quality_score=quality, anti_hack_violations=violations,
        constitutional_passed=const_passed, episode_valid=valid,
        steps=step_count,
    )

    if valid and const_passed and ep_briefs:
        try:
            traj = Trajectory(
                episode_num=ep_idx, scenario=scenario, wrr=wrr,
                brief_quality_score=quality, constitutional_passed=const_passed,
                episode_valid=valid, briefs=ep_briefs,
                reward_engine_snapshot=final_reward,
            )
            trajectory_buffer.add(traj)
        except Exception as e:
            print(f"  [!] Could not record trajectory for ep {ep_idx+1}: {e}")

    try:
        curriculum.record_episode(EpisodeResult(
            episode_num=ep_idx, scenario=scenario, wrr=wrr,
            brief_quality_score=quality, anti_hack_violations=violations,
            constitutional_passed=const_passed, episode_valid=valid,
        ))
    except Exception:
        pass

    print(
        f"{ep_idx+1:>4} {wrr:>6.3f} {r1:>6.3f} {r2:>6.3f} {r3:>6.3f} "
        f"{quality:>6.3f} {violations:>5} {'PASS' if const_passed else 'FAIL':>6} "
        f"{elapsed:>5.0f}s"
    )

print("-" * 60)
total_elapsed = time.time() - total_start

if episode_results:
    wrrs    = [e["wrr"] for e in episode_results]
    avg_wrr = sum(wrrs) / len(wrrs)
    max_wrr = max(wrrs)
    buf_size = trajectory_buffer.get_stats()["buffer_size"] if trajectory_buffer else 0

    print(f"\nGRPO Rollouts Summary")
    print(f"  Episodes completed  : {GRPO_EPISODES}")
    print(f"  Avg WRR             : {avg_wrr:.4f}")
    print(f"  Max WRR             : {max_wrr:.4f}")
    print(f"  Buffer size         : {buf_size}")
    print(f"  Total time          : {total_elapsed/60:.1f} min")
else:
    print("No episodes completed.")

log_path = os.path.join(WORK_DIR, "episode_log.json")
with open(log_path, "w") as f:
    json.dump(episode_results, f, indent=2)
print(f"\nEpisode log saved: {log_path}")


## Section 9 — DPO Fine-tuning (the actual RL gradient update)

Direct Preference Optimization. Loads the SFT model, builds chosen/rejected pairs from the trajectory buffer, and runs gradient updates with TRL's `DPOTrainer`. **This is the step that makes the agent learn from environment feedback.** Auto-skipped if VRAM is too small or the buffer is empty — falls back to the SFT checkpoint.


In [ ]:
# ============================================================
# CELL 12 — DPO FINE-TUNING (the actual RL gradient update)
#
# Steps performed:
#   1. Free SFT model + GRPO inference state from GPU (avoid OOM on T4).
#   2. Generate preference pairs from trajectory_buffer.
#   3. TRL DPOTrainer.train() — gradient updates against the SFT reference.
#   4. Save merged 16-bit checkpoint to DPO_DIR.
#   5. Print pre/post-DPO WRR delta so you can see if the agent improved.
#
# Auto-skipped if:
#   - DPO_ENABLED=False in Cell 1, or
#   - VRAM < DPO_MIN_VRAM_GB, or
#   - trajectory_buffer has < DPO_MIN_PAIRS clean episodes.
# ============================================================

import sys, os, gc, json, math, torch
sys.path.insert(0, REPO_DIR)

# Make wandb fully silent regardless of TRL's hardcoded report_to
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"]     = "disabled"

CURRENT_CHECKPOINT = SFT_DIR

def _decide_skip_reason():
    if not DPO_ENABLED:
        return "DPO_ENABLED=False in Cell 1"
    if VRAM_GB and VRAM_GB < DPO_MIN_VRAM_GB:
        return f"VRAM {VRAM_GB} GB < DPO_MIN_VRAM_GB ({DPO_MIN_VRAM_GB} GB)"
    if trajectory_buffer is None:
        return "trajectory_buffer is None (rollout cell did not run)"
    buf_size = trajectory_buffer.get_stats()["buffer_size"]
    if buf_size < DPO_MIN_PAIRS:
        return f"buffer has {buf_size} clean episodes, need >= {DPO_MIN_PAIRS}"
    return None

def _measure_wrr(checkpoint, n_episodes=2):
    """Quick WRR measurement on a checkpoint. Used for pre/post comparison."""
    from unsloth import FastLanguageModel
    from freshprice_env.freshprice_env import FreshPriceEnv
    from freshprice_env.enums import CurriculumScenario
    from freshprice_env.brief_pipeline.prompt_builder import OperatingBriefPromptBuilder

    m, t = FastLanguageModel.from_pretrained(
        model_name=checkpoint, max_seq_length=2048,
        dtype=None, load_in_4bit=True,
        token=HF_TOKEN if HF_TOKEN_SET else None,
    )
    FastLanguageModel.for_inference(m)
    sysp = OperatingBriefPromptBuilder.SYSTEM_PROMPT

    def _gen(prompt):
        full = (f"<|im_start|>system\n{sysp}<|im_end|>\n"
                f"<|im_start|>user\n{prompt}<|im_end|>\n"
                f"<|im_start|>assistant\n")
        ins = t(full, return_tensors="pt", truncation=True, max_length=3800).to(m.device)
        with torch.no_grad():
            out = m.generate(**ins, max_new_tokens=600, do_sample=False,
                             pad_token_id=t.eos_token_id)
        return t.decode(out[0][ins["input_ids"].shape[1]:], skip_special_tokens=True)

    class _Cli:
        def generate(self, p): return _gen(p)

    wrrs = []
    for i in range(n_episodes):
        env = FreshPriceEnv(scenario=CurriculumScenario.STABLE_WEEK,
                            seed=999 + i, llm_client=_Cli())
        obs, info = env.reset(seed=999 + i)
        done = False
        while not done:
            obs, r, done, _, info = env.step(_gen(obs))
        wrrs.append(float(info.get("final_reward", {}).get("wrr", 0.0)))

    del m, t
    gc.collect()
    torch.cuda.empty_cache()
    return wrrs

skip_reason = _decide_skip_reason()
if skip_reason:
    print(f"DPO skipped: {skip_reason}")
    print(f"Final checkpoint stays at SFT: {SFT_DIR}")
else:
    print("=" * 60)
    print("  RL GRADIENT UPDATE STARTING (DPO)")
    print("=" * 60)

    try:
        # ---- Free SFT model + inference state before DPO loads its own copy ----
        for var in ("model", "tokenizer", "trainer", "llm_client", "greedy_client"):
            if var in dir():
                try: exec(f"del {var}")
                except Exception: pass
        gc.collect(); torch.cuda.empty_cache()
        gc.collect(); torch.cuda.empty_cache()
        print(f"  GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated\n")

        # ---- (Optional) Pre-DPO WRR measurement ----
        # Skip on small GPUs to save time; T4 user can flip this on.
        MEASURE_PREPOST = (VRAM_GB >= 14)
        pre_wrrs = []
        if MEASURE_PREPOST:
            print("Measuring pre-DPO WRR (2 episodes)...")
            try:
                pre_wrrs = _measure_wrr(SFT_DIR, n_episodes=2)
                print(f"  Pre-DPO WRR per ep: {[f'{w:.4f}' for w in pre_wrrs]}")
                print(f"  Pre-DPO WRR mean  : {sum(pre_wrrs)/len(pre_wrrs):.4f}\n")
            except Exception as e:
                print(f"  Pre-DPO measurement skipped: {e}\n")
                pre_wrrs = []

        # ---- Build preference pairs ----
        print("Generating DPO preference pairs...")
        dpo_pairs = trajectory_buffer.generate_dpo_pairs(counterfactual_eng)
        print(f"  Generated {len(dpo_pairs)} DPO pairs.\n")

        if len(dpo_pairs) < DPO_MIN_PAIRS:
            print(f"DPO skipped: only {len(dpo_pairs)} pairs, need >= {DPO_MIN_PAIRS}.")
        else:
            from training.dpo_trainer import run_dpo
            os.makedirs(DPO_DIR, exist_ok=True)

            new_checkpoint = run_dpo(
                checkpoint_dir=SFT_DIR,
                output_dir=DPO_DIR,
                dpo_pairs=dpo_pairs,
                seed=SEED,
                learning_rate=DPO_LR_MANUAL,
                report_to="none",         # no wandb
                skip_verification=True,   # we'll do our own pre/post compare below
            )
            CURRENT_CHECKPOINT = new_checkpoint
            print(f"\nDPO complete. New checkpoint: {CURRENT_CHECKPOINT}")

            # ---- Post-DPO measurement and delta ----
            if MEASURE_PREPOST and pre_wrrs:
                gc.collect(); torch.cuda.empty_cache()
                print("\nMeasuring post-DPO WRR (2 episodes, same seeds as pre)...")
                try:
                    post_wrrs = _measure_wrr(CURRENT_CHECKPOINT, n_episodes=2)
                    print(f"  Post-DPO WRR per ep: {[f'{w:.4f}' for w in post_wrrs]}")
                    pre_mean = sum(pre_wrrs)/len(pre_wrrs)
                    post_mean = sum(post_wrrs)/len(post_wrrs)
                    delta = post_mean - pre_mean
                    print()
                    print("=" * 60)
                    print(f"  RL UPDATE RESULT (DPO):")
                    print(f"    Pre-DPO  mean WRR : {pre_mean:.4f}")
                    print(f"    Post-DPO mean WRR : {post_mean:.4f}")
                    print(f"    Delta             : {delta:+.4f} ({'IMPROVED' if delta>0 else 'NO IMPROVEMENT'})")
                    print("=" * 60)
                except Exception as e:
                    print(f"  Post-DPO measurement skipped: {e}")

    except torch.cuda.OutOfMemoryError as oom:
        print(f"[!] DPO OOM on this GPU: {oom}")
        print("    Falling back to SFT checkpoint.")
        CURRENT_CHECKPOINT = SFT_DIR
    except Exception as e:
        print(f"[!] DPO failed: {type(e).__name__}: {e}")
        print("    Falling back to SFT checkpoint.")
        CURRENT_CHECKPOINT = SFT_DIR

print(f"\nFinal checkpoint for evaluation: {CURRENT_CHECKPOINT}")


## Section 10 — Evaluation

In [ ]:
# ============================================================
# CELL 13 — DETERMINISTIC EVALUATION
# Greedy decoding on fixed seeds. Fully reproducible.
# Records every episode into the live monitoring dashboard.
# ============================================================

import sys, os, json, math, time, gc
sys.path.insert(0, REPO_DIR)

from freshprice_env.freshprice_env import FreshPriceEnv
from freshprice_env.enums import CurriculumScenario
from freshprice_env.monitoring import metrics
from freshprice_env.brief_pipeline.prompt_builder import OperatingBriefPromptBuilder

eval_results = {}

# If model/tokenizer were freed by Cell 12 (DPO), reload from CURRENT_CHECKPOINT.
need_reload = False
try:
    _ = model, tokenizer
except NameError:
    need_reload = True

if need_reload:
    print(f"Reloading model from {CURRENT_CHECKPOINT} for evaluation...")
    try:
        from unsloth import FastLanguageModel
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=CURRENT_CHECKPOINT,
            max_seq_length=2048,
            dtype=None,
            load_in_4bit=True,
            token=HF_TOKEN if HF_TOKEN_SET else None,
        )
    except Exception as e:
        print(f"[!] Could not reload model for eval: {e}")
        print("    Skipping evaluation.")
        model = None
        tokenizer = None

class GreedyClient:
    def __init__(self, model, tokenizer, system_prompt):
        from unsloth import FastLanguageModel
        FastLanguageModel.for_inference(model)
        self._model = model
        self._tok   = tokenizer
        self._sys   = system_prompt

    def generate(self, prompt: str) -> str:
        import torch
        full = (
            f"<|im_start|>system\n{self._sys}<|im_end|>\n"
            f"<|im_start|>user\n{prompt}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        inputs = self._tok(full, return_tensors="pt", truncation=True,
                           max_length=3800).to(self._model.device)
        with torch.no_grad():
            out = self._model.generate(
                **inputs, max_new_tokens=600,
                do_sample=False,
                pad_token_id=self._tok.eos_token_id,
            )
        return self._tok.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )

if model is None or tokenizer is None:
    print("Skipping evaluation (model not loaded).")
else:
    greedy_client = GreedyClient(
        model=model, tokenizer=tokenizer,
        system_prompt=OperatingBriefPromptBuilder.SYSTEM_PROMPT,
    )

    eval_scenarios = [
        CurriculumScenario.STABLE_WEEK,
        CurriculumScenario.CRISIS_WEEK,
    ]
    EVAL_SEEDS_PER_SCENARIO = 2  # keep small for speed; bump to 3+ for paper

    print("=" * 60)
    print(" EVALUATION REPORT")
    print(f" Checkpoint: {CURRENT_CHECKPOINT}")
    print("=" * 60)

    for scenario in eval_scenarios:
        level  = scenario.value
        seeds  = [level * 1000 + i for i in range(EVAL_SEEDS_PER_SCENARIO)]
        ep_res = []

        print(f"\n-- {scenario.name} ({EVAL_SEEDS_PER_SCENARIO} episodes) --")

        for seed in seeds:
            try:
                env  = FreshPriceEnv(scenario=scenario, seed=seed, llm_client=greedy_client)
                obs, info = env.reset(seed=seed)
                done = False
                while not done:
                    brief = greedy_client.generate(obs)
                    obs, reward, done, truncated, info = env.step(brief)

                fr    = info.get("final_reward", {}) or {}
                audit = info.get("constitutional_audit", {}) or {}

                rec = {
                    "seed": seed,
                    "wrr":     float(fr.get("wrr", 0.0)),
                    "r1":      float(fr.get("r1_pricing", 0.0)),
                    "r2":      float(fr.get("r2_farmer", 0.0)),
                    "r3":      float(fr.get("r3_trend", 0.0)),
                    "quality": float(fr.get("brief_quality_score", 0.0)),
                    "violations": int(fr.get("anti_hack_violations", 0)),
                    "constitutional_passed": bool(audit.get("passed", True)),
                }
                ep_res.append(rec)

                metrics.record_episode(
                    scenario=scenario.name, agent_type="llm-eval-greedy",
                    wrr=rec["wrr"], r1_pricing=rec["r1"], r2_farmer=rec["r2"], r3_trend=rec["r3"],
                    brief_quality_score=rec["quality"], anti_hack_violations=rec["violations"],
                    constitutional_passed=rec["constitutional_passed"], episode_valid=True,
                    steps=84,
                )

                print(f"  seed={seed}: WRR={rec['wrr']:.4f} | quality={rec['quality']:.4f} | "
                      f"violations={rec['violations']} | const={'PASS' if rec['constitutional_passed'] else 'FAIL'}")

            except Exception as e:
                print(f"  seed={seed}: FAILED ({type(e).__name__}: {e})")

        if not ep_res:
            print("  (no successful episodes)")
            continue

        wrrs = [e["wrr"] for e in ep_res]
        mean_wrr = sum(wrrs) / len(wrrs)
        std_wrr  = math.sqrt(sum((w - mean_wrr)**2 for w in wrrs) / max(len(wrrs)-1,1))

        eval_results[scenario.name] = {
            "wrr_mean": round(mean_wrr, 4),
            "wrr_std":  round(std_wrr, 4),
            "wrr_min":  round(min(wrrs), 4),
            "wrr_max":  round(max(wrrs), 4),
            "quality":  round(sum(e["quality"] for e in ep_res) / len(ep_res), 4),
            "violations_mean": round(sum(e["violations"] for e in ep_res) / len(ep_res), 1),
            "constitutional_pass_rate": f"{sum(e['constitutional_passed'] for e in ep_res)}/{len(ep_res)}",
        }

        r = eval_results[scenario.name]
        print(f"\n  WRR  : {r['wrr_mean']:.4f} +/- {r['wrr_std']:.4f}  [{r['wrr_min']:.4f} -> {r['wrr_max']:.4f}]")
        print(f"  Qual : {r['quality']:.4f}")
        print(f"  Viol : {r['violations_mean']} avg/episode")
        print(f"  Const: {r['constitutional_pass_rate']}")

eval_path = os.path.join(WORK_DIR, "eval_results.json")
with open(eval_path, "w") as f:
    json.dump(eval_results, f, indent=2)
print(f"\nEval results saved: {eval_path}")

if eval_results:
    all_wrrs = [v["wrr_mean"] for v in eval_results.values()]
    overall  = sum(all_wrrs) / len(all_wrrs)
    print(f"\nOverall mean WRR: {overall:.4f}")
    print(f"Promotion threshold: 0.70")
    if overall >= 0.70:
        print("STATUS: ABOVE PROMOTION THRESHOLD")
    else:
        print(f"STATUS: {0.70 - overall:.4f} below threshold — more training needed.")


## Section 11 — Anti-Hack Analysis

In [ ]:
# ============================================================
# CELL 14 — ANTI-HACK ANALYSIS
# Scans collected trajectories for 8 reward-hacking patterns.
# ============================================================

import sys
sys.path.insert(0, REPO_DIR)

from eval.anti_hack_checker import AntiHackChecker

print("Anti-Hack Pattern Analysis")
print("=" * 50)
print(f"Scanning {trajectory_buffer.get_stats()['buffer_size']} trajectories...")

top_trajectories = trajectory_buffer.get_top_n(n=50)

if not top_trajectories:
    print("Buffer is empty — no trajectories to scan.")
    print("(This is normal if GRPO_EPISODES was 0.)")
else:
    summary = AntiHackChecker.scan_trajectory_buffer(top_trajectories)

    print(f"\nSummary:")
    print(f"  Total scanned    : {summary['total_trajectories']}")
    print(f"  Clean (DPO-safe) : {summary['clean']}")
    print(f"  Flagged (review) : {summary['flagged_for_review']}")
    print(f"  Excluded (hack)  : {summary['excluded']}")

    if summary["pattern_frequency"]:
        print(f"\nDetected patterns:")
        for ptype, count in sorted(summary["pattern_frequency"].items(),
                                    key=lambda x: -x[1]):
            print(f"  {ptype:35s}: {count}")
        print(f"  Most common: {summary['most_common_pattern']}")
    else:
        print("  No hacking patterns detected.")

    dpo_safe_pct = summary['clean'] / summary['total_trajectories'] * 100
    print(f"\nDPO-safe rate: {dpo_safe_pct:.0f}%")
    if dpo_safe_pct < 50:
        print("WARNING: < 50% DPO-safe. Model may be reward hacking.")
        print("Consider more SFT epochs or reducing DPO learning rate.")

## Section 12 — Reward Curve Plots

In [ ]:
# ============================================================
# CELL 15 — REWARD CURVE PLOTS (graceful)
# ============================================================

import os, json
import matplotlib.pyplot as plt

log_path = os.path.join(WORK_DIR, "episode_log.json")
ep_log = []
if os.path.exists(log_path):
    try:
        with open(log_path) as f:
            ep_log = json.load(f)
    except Exception as e:
        print(f"Could not load {log_path}: {e}")

if len(ep_log) < 2:
    print(f"Only {len(ep_log)} episode(s) in log — skipping training-curve plot.")
else:
    episodes   = [e["episode"]+1 for e in ep_log]
    wrrs       = [e.get("wrr", 0.0)       for e in ep_log]
    r1s        = [e.get("r1", 0.0)        for e in ep_log]
    r2s        = [e.get("r2", 0.0)        for e in ep_log]
    r3s        = [e.get("r3", 0.0)        for e in ep_log]
    qualities  = [e.get("quality", 0.0)   for e in ep_log]
    violations = [e.get("violations", 0)  for e in ep_log]

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("QStorePrice AI — Training Metrics", fontsize=14, fontweight="bold")

    ax = axes[0, 0]
    ax.plot(episodes, wrrs, "b-o", linewidth=2, markersize=5, label="WRR")
    ax.axhline(y=0.70, color="green", linestyle="--", alpha=0.8, label="Promotion (0.70)")
    ax.axhline(y=0.08, color="red",   linestyle=":",  alpha=0.6, label="Zero-shot baseline (~0.08)")
    ax.fill_between(episodes, wrrs, 0.08, alpha=0.15, color="blue")
    ax.set_xlabel("Episode"); ax.set_ylabel("WRR"); ax.set_title("Weekly Waste Recovery Rate")
    ax.legend(fontsize=8); ax.set_ylim(-0.05, 1.05); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(episodes, r1s, "r-^", markersize=4, label="r1 Pricing (w=0.40)")
    ax.plot(episodes, r2s, "g-s", markersize=4, label="r2 Farmer (w=0.30)")
    ax.plot(episodes, r3s, "b-D", markersize=4, label="r3 Trend (w=0.30)")
    ax.set_xlabel("Episode"); ax.set_ylabel("Reward"); ax.set_title("Reward Component Breakdown")
    ax.legend(fontsize=8); ax.set_ylim(-0.1, 1.1); ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    ax.plot(episodes, qualities, "m-o", linewidth=2, markersize=5, label="Brief Quality")
    ax.set_xlabel("Episode"); ax.set_ylabel("Quality Score (0-1)")
    ax.set_title("Brief Quality Score (independent of WRR)")
    ax.set_ylim(0, 1.05)
    ax.axhline(y=0.7, color="orange", linestyle="--", alpha=0.6, label="Quality target")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    colors = ["red" if v > 3 else "orange" if v > 1 else "green" for v in violations]
    ax.bar(episodes, violations, color=colors, alpha=0.7)
    ax.set_xlabel("Episode"); ax.set_ylabel("Violation Count")
    ax.set_title("Anti-Hack Violations per Episode")
    ax.axhline(y=3, color="red", linestyle="--", alpha=0.5, label="Warning threshold")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plot1_path = os.path.join(PLOTS_DIR, "training_metrics.png")
    plt.savefig(plot1_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Plot saved: {plot1_path}")

# ---- Eval bar chart ----
if eval_results:
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    sc_names  = list(eval_results.keys())
    sc_wrrs   = [eval_results[s]["wrr_mean"] for s in sc_names]
    sc_stds   = [eval_results[s]["wrr_std"]  for s in sc_names]
    bar_colors = ["green" if w >= 0.70 else "steelblue" for w in sc_wrrs]

    bars = ax2.bar(sc_names, sc_wrrs, color=bar_colors, alpha=0.8,
                   yerr=sc_stds, capsize=5, error_kw={"linewidth": 2})
    ax2.axhline(y=0.70, color="green",  linestyle="--", alpha=0.8, label="Promotion (0.70)")
    ax2.axhline(y=0.08, color="red",    linestyle=":",  alpha=0.6, label="Zero-shot baseline")

    for bar, wrr in zip(bars, sc_wrrs):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{wrr:.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

    ax2.set_ylabel("WRR (Weekly Waste Recovery Rate)")
    ax2.set_title("Evaluation WRR per Scenario")
    ax2.set_ylim(0, 1.0)
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plot2_path = os.path.join(PLOTS_DIR, "eval_wrr_by_scenario.png")
    plt.savefig(plot2_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Plot saved: {plot2_path}")
else:
    print("No eval_results — skipping eval bar chart.")


## Section 13 — Backend Server

In [ ]:
# ============================================================
# CELL 16 — START FASTAPI SERVER (background, graceful)
# ============================================================

import sys, os, subprocess, time
sys.path.insert(0, REPO_DIR)

server_proc = None
server_up = False

def _start_server():
    try:
        proc = subprocess.Popen(
            ["python", "-m", "server.app"],
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            env={**os.environ, "PORT": str(SERVER_PORT), "WORKERS": "1"},
        )
        return proc
    except Exception as e:
        print(f"Could not spawn server process: {e}")
        return None

def _wait_for_server(url, retries=8, delay=2):
    import urllib.request
    for i in range(retries):
        try:
            with urllib.request.urlopen(f"{url}/health", timeout=3) as r:
                if r.status == 200:
                    return True
        except Exception:
            pass
        time.sleep(delay)
        print(f"  waiting for server... ({i+1}/{retries})")
    return False

print("Starting FastAPI backend server...")
server_proc = _start_server()

if server_proc is None:
    print("Server not started; downstream cells will use inline simulation.")
else:
    print(f"Server PID={server_proc.pid}. Probing /health...")
    server_up = _wait_for_server(SERVER_URL)
    if not server_up:
        try:
            err = server_proc.stderr.read().decode()[-1000:] if server_proc.stderr else ""
        except Exception:
            err = ""
        print(f"Server did not become healthy within timeout.")
        if err:
            print(f"Last stderr lines:\n{err}")
        try:
            server_proc.terminate()
        except Exception:
            pass
        server_proc = None
    else:
        print(f"Server is UP at {SERVER_URL}")


In [ ]:
# ============================================================
# CELL 17 — TEST ALL BACKEND ENDPOINTS
# Tests /health, /reset, /step, /state in sequence.
# If server is unavailable, uses a pure-Python simulation.
# ============================================================

import json, urllib.request, urllib.error

def http_get(url):
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            return json.loads(r.read())
    except Exception as e:
        return {"error": str(e)}

def http_post(url, data):
    try:
        req = urllib.request.Request(
            url, data=json.dumps(data).encode(), method="POST",
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(req, timeout=30) as r:
            return json.loads(r.read())
    except Exception as e:
        return {"error": str(e)}


if server_up:
    print("=" * 50)
    print(" BACKEND API TEST — Live Server")
    print("=" * 50)

    # 1. Health check
    resp = http_get(f"{SERVER_URL}/health")
    print(f"\nGET /health        → {resp}")

    # 2. Reset episode
    resp = http_post(f"{SERVER_URL}/reset", {"scenario": "STABLE_WEEK", "seed": 42})
    print(f"POST /reset        → keys: {list(resp.keys())}")
    obs_excerpt = str(resp.get("observation", ""))[:200]
    print(f"  observation[:200]: {obs_excerpt}")

    # 3. Step with a minimal brief
    step_payload = {
        "brief_text": (
            "## SITUATION SUMMARY\nBatch analysis complete.\n\n"
            "## SIGNAL ANALYSIS\nModerate expiry risk.\n\n"
            "## VIABILITY CHECK\nDiscount viable.\n\n"
            "## RECOMMENDATION\nHold prices on FRESH stock.\n\n"
            '## DIRECTIVE\n{"engine": "PRICING", "actions": []}\n\n'
            "## CONFIDENCE\n0.65\n"
        )
    }
    resp = http_post(f"{SERVER_URL}/step", step_payload)
    print(f"POST /step         → keys: {list(resp.keys())}")
    print(f"  reward           : {resp.get('reward', 'N/A')}")
    print(f"  done             : {resp.get('done', 'N/A')}")
    info_keys = list(resp.get('info', {}).keys())
    print(f"  info keys        : {info_keys}")

    # 4. State
    resp = http_get(f"{SERVER_URL}/state")
    print(f"GET /state         → {json.dumps(resp, indent=2)[:500]}")

    # 5. Docs endpoint
    resp = http_get(f"{SERVER_URL}/docs")
    print(f"GET /docs          → {'HTML page available' if 'error' not in resp else resp}")

    print(f"\nAll API endpoints tested. Server URL: {SERVER_URL}")
    print(f"Swagger UI: {SERVER_URL}/docs")

else:
    print("=" * 50)
    print(" BACKEND API — Python Simulation (no live server)")
    print("=" * 50)
    print("Server did not start (likely missing openenv-core).")
    print("Demonstrating equivalent logic via direct Python calls:")

    import sys
    sys.path.insert(0, REPO_DIR)
    from freshprice_env.freshprice_env import FreshPriceEnv
    from freshprice_env.enums import CurriculumScenario

    # Simulate GET /health
    print('\nSimulated GET /health → {"status": "ok"}')

    # Simulate POST /reset
    env = FreshPriceEnv(scenario=CurriculumScenario.STABLE_WEEK, seed=42)
    obs, info = env.reset()
    print(f"Simulated POST /reset → observation length={len(obs)}, engine={info['engine_type']}")

    # Simulate POST /step
    test_brief = (
        "## SITUATION SUMMARY\nInventory assessed.\n\n"
        "## SIGNAL ANALYSIS\nModerate demand.\n\n"
        "## VIABILITY CHECK\nConservative strategy.\n\n"
        "## RECOMMENDATION\nHold prices.\n\n"
        '## DIRECTIVE\n{"engine": "PRICING", "actions": []}\n\n'
        "## CONFIDENCE\n0.6\n"
    )
    obs2, reward, done, truncated, info2 = env.step(test_brief)
    print(f"Simulated POST /step → reward={reward:.4f}, done={done}")
    print(f"  parse_success={info2['parse_success']}, quality={info2['quality_score']:.3f}")

    # Simulate GET /state
    state = env.state()
    print(f"Simulated GET /state → {json.dumps(state, indent=2)[:400]}")

    print("\nTo run the live server locally:")
    print("  pip install openenv-core")
    print(f"  cd {REPO_DIR} && python -m server.app")

In [ ]:
# ============================================================
# CELL 18 — STOP SERVER (cleanup)
# ============================================================

try:
    if server_proc is not None:
        server_proc.terminate()
        try:
            server_proc.wait(timeout=5)
        except Exception:
            server_proc.kill()
        print(f"Server process {server_proc.pid} terminated.")
    else:
        print("No server process to terminate.")
except NameError:
    print("server_proc not defined; nothing to terminate.")


## Section 14 — Push to Hugging Face Hub

In [ ]:
# ============================================================
# CELL 19 — PUSH TRAINED MODEL TO HUGGING FACE HUB
# Skipped if HF_TOKEN was left as the placeholder.
# ============================================================

import os, json

if not HF_TOKEN_SET or not HF_AUTHED:
    print("HF push skipped: HF_TOKEN not set or auth failed.")
    print("  - Set HF_TOKEN in Cell 1 (or via Kaggle Secrets) to enable pushing.")
elif "your-hf-username" in HF_REPO_ID:
    print(f"HF push skipped: HF_REPO_ID still placeholder ({HF_REPO_ID}).")
    print("  - Set HF_REPO_ID in Cell 1 to a real `<your-username>/<repo>` value.")
else:
    try:
        from huggingface_hub import HfApi, upload_folder

        api = HfApi(token=HF_TOKEN)
        print(f"Creating/verifying HF repo: {HF_REPO_ID}")
        api.create_repo(
            repo_id=HF_REPO_ID, repo_type="model",
            exist_ok=True, private=False,
        )

        push_dir = CURRENT_CHECKPOINT if os.path.exists(CURRENT_CHECKPOINT) else SFT_DIR
        print(f"Pushing from: {push_dir}")

        model_card = f"""---
tags:
  - qstoreprice
  - reinforcement-learning
  - perishable-goods
  - operating-brief
  - wrr
base_model: {MODEL_ID}
---

# QStorePrice AI — Trained Checkpoint

**Base model**: `{MODEL_ID}`
**Training**: SFT warm-start ({SFT_EPOCHS} epochs) + GRPO rollouts ({GRPO_EPISODES} episodes)
**Metric**: WRR (Weekly Waste Recovery Rate)

## Evaluation Results

```json
{json.dumps(eval_results, indent=2)}
```

## Project
- GitHub: https://github.com/nandeshkanagaraju/QStorePrice
- Trained on Kaggle with Unsloth 4-bit + LoRA
"""
        card_path = os.path.join(push_dir, "README.md")
        with open(card_path, "w") as f:
            f.write(model_card)

        upload_folder(
            repo_id=HF_REPO_ID,
            folder_path=push_dir,
            repo_type="model",
            token=HF_TOKEN,
            ignore_patterns=["*.pyc", "__pycache__", "optimizer.pt"],
            commit_message=f"QStorePrice SFT+GRPO checkpoint",
        )
        print(f"Model pushed: https://huggingface.co/{HF_REPO_ID}")

        for plot_file in ["training_metrics.png", "eval_wrr_by_scenario.png"]:
            plot_path = os.path.join(PLOTS_DIR, plot_file)
            if os.path.exists(plot_path):
                try:
                    api.upload_file(
                        path_or_fileobj=plot_path,
                        path_in_repo=f"plots/{plot_file}",
                        repo_id=HF_REPO_ID, repo_type="model", token=HF_TOKEN,
                    )
                    print(f"  Uploaded: {plot_file}")
                except Exception as e:
                    print(f"  {plot_file}: {e}")

    except Exception as e:
        print(f"HF push failed: {type(e).__name__}: {e}")


## Section 15 — Final Summary

In [ ]:
# ============================================================
# CELL 20 — FINAL SUMMARY
# ============================================================

import os, json

print("=" * 65)
print(" QStorePrice AI — Run Complete")
print("=" * 65)

print(f"\n{'Model':<30}: {MODEL_ID}")
print(f"{'SFT epochs':<30}: {SFT_EPOCHS}")
print(f"{'GRPO episodes':<30}: {GRPO_EPISODES}")
print(f"{'DPO enabled':<30}: {DPO_ENABLED}")
print(f"{'Seed':<30}: {SEED}")
print(f"{'Final checkpoint':<30}: {CURRENT_CHECKPOINT}")
if HF_TOKEN_SET and "your-hf-username" not in HF_REPO_ID:
    print(f"{'HF repo':<30}: https://huggingface.co/{HF_REPO_ID}")
else:
    print(f"{'HF repo':<30}: (skipped)")

if eval_results:
    print(f"\n{'-'*65}")
    print(" Evaluation Results")
    print(f"{'-'*65}")
    print(f"  {'Scenario':<22} {'WRR':>8} {'+/-':>6} {'Quality':>8} {'Viol':>5} {'Const':>6}")
    print(f"  {'-'*56}")
    for sc, r in eval_results.items():
        print(
            f"  {sc:<22} {r['wrr_mean']:>8.4f} {r['wrr_std']:>6.4f} "
            f"{r['quality']:>8.4f} {r['violations_mean']:>5.1f} "
            f"{r['constitutional_pass_rate']:>6}"
        )
    all_wrrs = [v["wrr_mean"] for v in eval_results.values()]
    overall  = sum(all_wrrs) / len(all_wrrs)
    print(f"  {'-'*56}")
    print(f"  {'Overall mean WRR':<22} {overall:>8.4f}")
else:
    print("\n(no eval results)")

print(f"\n{'-'*65}")
print(" Output Files")
print(f"{'-'*65}")
outputs = [
    ("SFT checkpoint",         SFT_DIR),
    ("DPO dir",                DPO_DIR),
    ("Episode log",            f"{WORK_DIR}/episode_log.json"),
    ("Eval results",           f"{WORK_DIR}/eval_results.json"),
    ("Training metrics plot",  f"{PLOTS_DIR}/training_metrics.png"),
    ("Eval WRR plot",          f"{PLOTS_DIR}/eval_wrr_by_scenario.png"),
]
for label, path in outputs:
    exists = os.path.exists(path)
    size_str = ""
    if exists and os.path.isdir(path):
        total = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, fns in os.walk(path) for f in fns
        )
        size_str = f"({total/1e6:.0f} MB)"
    elif exists:
        size_str = f"({os.path.getsize(path)/1e3:.0f} KB)"
    status = f"OK {size_str}" if exists else "(skipped/missing)"
    print(f"  {label:<30}: {status}")

print("\n" + "=" * 65)
print(" Run complete.")
print("=" * 65)


## Section 16 — Live Admin Dashboard
Hits `/admin/dashboard` (added by the ported features) and renders the metrics inline.

In [ ]:
# ============================================================
# CELL 21 — POLL ADMIN DASHBOARD
# Brings the live metrics into the notebook so judges can see WRR,
# quality, violations, etc. without leaving Kaggle.
# ============================================================

import json, urllib.request

print("Admin dashboard snapshot")
print("=" * 60)

if not server_up:
    print("Server is not running — pulling metrics directly from the in-process store.")
    try:
        from freshprice_env.monitoring import metrics
        snap = metrics.get_dashboard()
    except Exception as e:
        print(f"Could not load monitoring module: {e}")
        snap = {}
else:
    try:
        with urllib.request.urlopen(f"{SERVER_URL}/admin/dashboard", timeout=5) as r:
            snap = json.loads(r.read())
    except Exception as e:
        print(f"GET /admin/dashboard failed: {e}; falling back to in-process store.")
        from freshprice_env.monitoring import metrics
        snap = metrics.get_dashboard()

if not snap:
    print("(no metrics recorded)")
else:
    s = snap.get("summary", {})
    print(f"  Episodes total          : {s.get('episodes_total', 0)}")
    print(f"  Steps total             : {s.get('steps_total', 0)}")
    print(f"  WRR mean / max          : {s.get('wrr_mean', 0):.4f} / {s.get('wrr_max', 0):.4f}")
    print(f"  Brief quality mean      : {s.get('quality_mean', 0):.4f}")
    print(f"  Anti-hack violations    : {s.get('violations_total', 0)}")
    pass_rate = s.get('constitutional_pass_rate', 1.0)
    print(f"  Constitutional pass rate: {pass_rate*100:.0f}%")

    by_sc = snap.get("by_scenario", {})
    if by_sc:
        print("\n  Per scenario:")
        for name, b in by_sc.items():
            print(f"    {name:<15} n={b.get('n',0):<3} WRR={b.get('wrr_mean',0):.4f}")

    print(f"\n  Recent episodes ({len(snap.get('recent_episodes', []))}):")
    for ep in snap.get("recent_episodes", [])[-5:]:
        const = "PASS" if ep.get("constitutional_passed", True) else "FAIL"
        print(f"    {ep.get('scenario','?'):<15} agent={ep.get('agent_type','?'):<18} "
              f"WRR={ep.get('wrr',0):.4f} viol={ep.get('anti_hack_violations',0)} const={const}")

print("\nFull JSON also at: " + (f"{SERVER_URL}/admin/dashboard" if server_up else "(in-process only)"))
